# DocuMind RAG Evaluation with LangSmith — Single Run

This notebook evaluates the DocuMind RAG implementation in `src/` against the complete 30-question dataset in **one LangSmith experiment**. It uses four LLM-as-judge metrics:

- **Correctness:** Is the answer factually consistent with the reference answer?
- **Answer relevance:** Does the answer address the question?
- **Groundedness:** Is the answer supported by retrieved document content?
- **Retrieval relevance:** Is the retrieved context relevant to the question?

## Before you start

1. Run this notebook from the DocuMind project root.
2. Make sure `data/` contains the source PDFs and `faiss_store/` was built from those same documents.
3. Add `GROQ_API_KEY` and `LANGSMITH_API_KEY` to your `.env` file.
4. Review the reference answers before interpreting the scores.

This version submits the full dataset in one evaluation run. It is simpler, but it may take longer and be more susceptible to provider rate limits than the batch notebook. The evaluator implementation shown here uses Groq; the judge model/provider can be replaced by adapting `GroqStructuredEvaluator` and the structured-output interface. The DocuMind answer-generation model is configured separately in `src/search_updated.py`.

LLM-as-judge scores are diagnostic estimates, not ground-truth measurements.


In [ ]:
from langchain_groq import ChatGroq
import os

EVALUATOR_MODEL = "openai/gpt-oss-120b"

class GroqStructuredEvaluator:
    """Small wrapper for structured LLM-as-judge responses."""

    def __init__(self, schema, api_key=None):
        api_key = api_key or os.getenv("GROQ_API_KEY")
        if not api_key:
            raise EnvironmentError("Set GROQ_API_KEY in your .env file or environment.")
        self.llm = ChatGroq(
            model=EVALUATOR_MODEL,
            temperature=0,
            max_retries=3,
            api_key=api_key,
        ).with_structured_output(schema)

    def invoke(self, messages):
        return self.llm.invoke(messages)

print(f"Evaluator model: {EVALUATOR_MODEL}")


## Model and API configuration

Groq is used for the evaluation judges. The DocuMind answer-generation model is configured separately in `src/search_updated.py` (or the module used by your project). Keep API keys in `.env`; never commit that file.

## 1. Configure environment and imports

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Ensure imports and relative paths resolve from the DocuMind project root.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError(
        f"Open/run this notebook from the DocuMind project root. Current directory: {PROJECT_ROOT}"
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

required = ["LANGSMITH_API_KEY", "GROQ_API_KEY"]
missing = [key for key in required if not os.getenv(key)]
if missing:
    raise EnvironmentError(
        "Missing environment variables in .env or environment: " + ", ".join(missing)
    )

os.environ.setdefault("LANGSMITH_TRACING", "true")
from langsmith import Client, traceable
client = Client()
print("Project root:", PROJECT_ROOT)
print("LangSmith client initialized.")


## 2. Create or reuse the evaluation dataset

In [ ]:
DATASET_NAME = "DocuMind src RAG Evaluation - 30 Questions v1"
DATASET_DESCRIPTION = (
    "30-question evaluation set for DocuMind src/ implementation: Machine Learning, "
    "DevOps, cross-document, and unanswerable questions. Verify draft references "
    "against source PDFs before scoring."
)


examples = [
    {
        "inputs": {
            "question": "What is supervised learning?"
        },
        "outputs": {
            "answer": "Supervised learning trains a model using labeled examples, where each input is paired with a known target or output."
        }
    },
    {
        "inputs": {
            "question": "What is unsupervised learning?"
        },
        "outputs": {
            "answer": "Unsupervised learning uses data without supplied target labels and attempts to discover structure or patterns, for example through clustering."
        }
    },
    {
        "inputs": {
            "question": "What is the difference between supervised and unsupervised learning?"
        },
        "outputs": {
            "answer": "Supervised learning learns from labeled input-output examples. Unsupervised learning uses inputs without target labels to discover structure or patterns."
        }
    },
    {
        "inputs": {
            "question": "What is the k-Nearest Neighbors (k-NN) algorithm?"
        },
        "outputs": {
            "answer": "k-NN finds the k closest training examples to a new example. For classification it uses their class labels; for regression it combines their target values."
        }
    },
    {
        "inputs": {
            "question": "How does k-NN make a classification prediction?"
        },
        "outputs": {
            "answer": "It finds the k nearest training examples and predicts the class most commonly represented among those neighbors."
        }
    },
    {
        "inputs": {
            "question": "What happens when k = 1 in k-NN?"
        },
        "outputs": {
            "answer": "The prediction is based on the single nearest training example; for classification, that example's class is predicted."
        }
    },
    {
        "inputs": {
            "question": "What is Ridge regression?"
        },
        "outputs": {
            "answer": "Ridge regression is a linear regression method that adds an L2 penalty to coefficient values to discourage large coefficients and help control overfitting."
        }
    },
    {
        "inputs": {
            "question": "Why is preprocessing important in a machine-learning workflow?"
        },
        "outputs": {
            "answer": "Preprocessing prepares raw data for modeling. Depending on the data, it may include handling missing values, encoding categorical features, scaling numerical features, or transforming text."
        }
    },
    {
        "inputs": {
            "question": "What is the difference between stemming and lemmatization?"
        },
        "outputs": {
            "answer": "Stemming applies simple rules to reduce a word to a stem that may not be a real word. Lemmatization uses vocabulary and often grammatical information to return a canonical dictionary form."
        }
    },
    {
        "inputs": {
            "question": "What is grid search used for in machine learning?"
        },
        "outputs": {
            "answer": "Grid search evaluates combinations from a specified set of hyperparameter values, often with cross-validation, to identify a well-performing configuration."
        }
    },
    {
        "inputs": {
            "question": "What is clustering?"
        },
        "outputs": {
            "answer": "Clustering is an unsupervised learning task that groups observations based on similarity without relying on predefined target labels."
        }
    },
    {
        "inputs": {
            "question": "Explain k-means clustering and one limitation mentioned in the document."
        },
        "outputs": {
            "answer": "K-means partitions observations into k clusters by assigning points to cluster centers and iteratively updating the centers. A possible limitation is that k must be chosen in advance, if this is supported by the source document."
        }
    },
    {
        "inputs": {
            "question": "What is DBSCAN, and how does it handle noise?"
        },
        "outputs": {
            "answer": "DBSCAN is a density-based clustering algorithm. Points that do not meet its density criteria may be labeled as noise or outliers."
        }
    },
    {
        "inputs": {
            "question": "How can clustering be used to explore or visualize data?"
        },
        "outputs": {
            "answer": "Clustering can reveal groups or structure in observations and help explore similarities in a dataset. Visualization claims should be supported by the source document."
        }
    },
    {
        "inputs": {
            "question": "What is dimensionality reduction, and why might it be useful?"
        },
        "outputs": {
            "answer": "Dimensionality reduction represents data with fewer dimensions while aiming to preserve useful structure. It may simplify data or support visualization where the source document supports these uses."
        }
    },
    {
        "inputs": {
            "question": "What does DevOps mean?"
        },
        "outputs": {
            "answer": "DevOps brings software development and IT operations closer through collaboration, shared responsibility, and practices supporting reliable software delivery."
        }
    },
    {
        "inputs": {
            "question": "What are the main benefits of DevOps described in the handbook?"
        },
        "outputs": {
            "answer": "The answer should identify the main benefits explicitly described in the handbook, such as faster delivery, improved collaboration, feedback, reliability, or quality, where supported by the source."
        }
    },
    {
        "inputs": {
            "question": "How does DevOps improve time to market?"
        },
        "outputs": {
            "answer": "DevOps can shorten delivery cycles through improved collaboration and automation of repeatable build, test, release, and deployment activities, where these mechanisms are supported by the handbook."
        }
    },
    {
        "inputs": {
            "question": "How does DevOps support continuous integration and continuous delivery or deployment?"
        },
        "outputs": {
            "answer": "Continuous integration frequently integrates changes and validates them with automated builds and tests. Continuous delivery or deployment automates release steps; the explanation should follow the handbook's terminology."
        }
    },
    {
        "inputs": {
            "question": "What is DevSecOps?"
        },
        "outputs": {
            "answer": "DevSecOps integrates security practices into the software development and delivery lifecycle rather than treating security only as a final-stage activity."
        }
    },
    {
        "inputs": {
            "question": "How can automated security checks support a DevOps workflow?"
        },
        "outputs": {
            "answer": "Automated security checks can be integrated into development and CI/CD workflows to detect issues earlier and provide repeatable feedback before release, where supported by the handbook."
        }
    },
    {
        "inputs": {
            "question": "What roles do communication and collaboration play in DevOps?"
        },
        "outputs": {
            "answer": "Communication and collaboration help development and operations share information and responsibility, reduce silos, and coordinate changes and feedback."
        }
    },
    {
        "inputs": {
            "question": "What future outlook or industry adoption of DevOps does the handbook describe?"
        },
        "outputs": {
            "answer": "The answer should summarize only future outlook, adoption trends, or examples explicitly stated in the handbook. If the document does not address this, it should state that sufficient information is unavailable."
        }
    },
    {
        "inputs": {
            "question": "Compare supervised learning with DevOps."
        },
        "outputs": {
            "answer": "Supervised learning learns from labeled input-output examples. DevOps is a software delivery and operations approach focused on collaboration and reliable delivery. Each description should be supported by its respective source."
        }
    },
    {
        "inputs": {
            "question": "Compare the benefits of DevOps with the applications of clustering."
        },
        "outputs": {
            "answer": "DevOps benefits relate to software delivery and collaboration, while clustering groups similar data and helps explore structure. Support the DevOps explanation with the handbook and the clustering explanation with the machine-learning PDF."
        }
    },
    {
        "inputs": {
            "question": "Explain k-NN and DevSecOps in one answer."
        },
        "outputs": {
            "answer": "k-NN predicts using the closest training examples; classification typically uses their labels. DevSecOps integrates security into software development and delivery. Each explanation should be supported by its respective source."
        }
    },
    {
        "inputs": {
            "question": "How do machine-learning preprocessing and DevOps practices differ in their goals?"
        },
        "outputs": {
            "answer": "Machine-learning preprocessing prepares data for modeling. DevOps practices improve collaboration and the software delivery and operations lifecycle. Support each explanation with its respective source."
        }
    },
    {
        "inputs": {
            "question": "Compare the role of automation in DevOps with the use of algorithms in clustering."
        },
        "outputs": {
            "answer": "DevOps automation supports repeatable software delivery or operational activities, while clustering algorithms identify groups or patterns in data. Explain their different purposes using evidence from both documents."
        }
    },
    {
        "inputs": {
            "question": "What is the author's home address?"
        },
        "outputs": {
            "answer": "The provided Machine Learning and DevOps documents do not provide the author's home address, so it cannot be determined from the available sources."
        }
    },
    {
        "inputs": {
            "question": "What is the author's current salary?"
        },
        "outputs": {
            "answer": "The provided Machine Learning and DevOps documents do not state the author's current salary, so it cannot be determined from the available sources."
        }
    }
]


# Reuse the dataset on notebook reruns; don't duplicate its examples.
if client.has_dataset(dataset_name=DATASET_NAME):
    dataset = client.read_dataset(dataset_name=DATASET_NAME)
    print(f"Using existing dataset: {DATASET_NAME} ({dataset.id})")
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description=DATASET_DESCRIPTION,
    )
    client.create_examples(dataset_id=dataset.id, examples=examples)
    print(f"Created dataset: {DATASET_NAME} ({dataset.id})")

print("Expected example count:", len(examples))


## 3. Load the DocuMind RAG implementation

In [ ]:
from src.search_updated import RAGSearch

# Uses the FAISS index and metadata used by main.py. If either file is missing,
# RAGSearch builds the index from data/ using src/data_loader.py and src/embedding.py.
PERSIST_DIR = str(PROJECT_ROOT / "faiss_store")
TOP_K = 5
rag_search = RAGSearch(persist_dir=PERSIST_DIR)

print("RAG implementation: src.search_updated.RAGSearch")
print("Vector store:", PERSIST_DIR)
print("Retrieval top_k:", TOP_K)


## 4. Define the target function

In [ ]:
@traceable(name="documind-src-rag")
def app_target(inputs: dict) -> dict:
    """Run one question through DocuMind's src/ implementation.

    The existing search_with_sources method returns (answer, sources), where
    each source is a dict containing text, name, page, and label. The evaluators
    below consume these dictionaries directly.
    """
    question = inputs["question"]
    answer, sources = rag_search.search_with_sources(question, top_k=TOP_K)
    return {"answer": answer, "documents": sources}

# Smoke test one question before running the full evaluation.
smoke = app_target({"question": "What is supervised learning?"})
print(smoke["answer"])
print("Retrieved source chunks:", len(smoke["documents"]))


## 5. Define evaluation metrics

In [ ]:
from typing_extensions import Annotated,TypedDict

## Correctness Output Schema

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

## correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Provide a concise explanation for the score."""


grader_llm = GroqStructuredEvaluator(CorrectnessGrade)
## evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])
    return grade["correct"]







### Answer relevance

In [ ]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Provide a concise explanation for the score."""

# Grader LLM
relevance_llm = GroqStructuredEvaluator(RelevanceGrade)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### Groundedness

In [ ]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Provide a concise explanation for the score."""

# Grader LLM 
grounded_llm = GroqStructuredEvaluator(GroundedGrade)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.get("text", "") for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

### Retrieval relevance

In [ ]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) Your goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Provide a concise explanation for the score."""

# Grader LLM
retrieval_relevance_llm = GroqStructuredEvaluator(RetrievalRelevanceGrade)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.get("text", "") for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

## 6. Run the evaluation

This runs the RAG pipeline for the full 30-question dataset in one LangSmith experiment and calls four LLM judges per question. It is straightforward to inspect and manage, but may take several minutes and uses Groq and LangSmith API quota. If you encounter provider rate limits, use the separate batch notebook or reduce the workload.


In [ ]:
## Create or reuse the full evaluation dataset

if client.has_dataset(dataset_name=DATASET_NAME):
    evaluation_dataset = client.read_dataset(dataset_name=DATASET_NAME)
    print(f"Using existing dataset: {DATASET_NAME}")
else:
    evaluation_dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description=DATASET_DESCRIPTION,
    )
    client.create_examples(dataset_id=evaluation_dataset.id, examples=examples)
    print(f"Created dataset: {DATASET_NAME}")

print("Examples in source list:", len(examples))
print("Dataset ID:", evaluation_dataset.id)


## Run all 30 questions in one experiment

from langsmith import evaluate

evaluators = [
    correctness,
    groundedness,
    relevance,
    retrieval_relevance,
]

common_metadata = {
    "implementation": "src.search_updated.RAGSearch",
    "embedding_model": "all-MiniLM-L6-v2",
    "vector_store": "FAISS IndexFlatL2",
    "top_k": TOP_K,
    "evaluation_mode": "single-run",
}

print(f"Evaluating all {len(examples)} questions in one run...")

results = client.evaluate(
    app_target,
    data=DATASET_NAME,
    evaluators=evaluators,
    experiment_prefix="documind-rag-evaluation-all-30",
    metadata=common_metadata,
)

results_df = results.to_pandas()
print("Completed examples:", len(results_df))
display(results_df.head())


## 7. Review and export results

In [ ]:
# Inspect available metric columns and summary statistics.
print(results_df.columns.tolist())
display(results_df.describe(include="all").transpose())

# Export a local copy for your project evidence.
results_path = PROJECT_ROOT / "evaluation" / "langsmith_src_rag_results_single_run.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(results_path, index=False)
print("Saved results to:", results_path)
